# Notebook 07: Qualitative and Category Error Analysis

### Project: Efficient Financial Sentiment Fine-Tuning with LoRA and QLoRA

A strong research project doesn't just report aggregate metrics like F1—it investigates **why** models make errors.

In this notebook, we analyze the actual misclassifications made by:
1. **Zero-Shot Baseline**
2. **LoRA (16-bit)**
3. **QLoRA (4-bit)**

### Error Categories Studied:
- Positive $\rightarrow$ Neutral
- Positive $\rightarrow$ Negative
- Negative $\rightarrow$ Neutral
- Negative $\rightarrow$ Positive
- Neutral $\rightarrow$ Positive
- Neutral $\rightarrow$ Negative

In [1]:
# === MOUNT GOOGLE DRIVE & SETUP ===
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')

os.chdir('/content/drive/MyDrive/llm-project/lora-qlora-financial-sentiment')
sys.path.insert(0, os.getcwd())

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'datasets', 'peft', 'accelerate', 'bitsandbytes',
    'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'pyyaml'],
    check=True)

for d in ['results/metrics', 'results/predictions', 'results/figures',
          'results/logs', 'adapters/lora', 'adapters/qlora']:
    os.makedirs(d, exist_ok=True)

print(f'Working directory: {os.getcwd()}')
print(f'src/ found: {os.path.exists("src")}')  # Must be True!
print('Setup complete! ✅')

Mounted at /content/drive
Working directory: /content/drive/.shortcut-targets-by-id/1fmKQoCL8RLydtggUMrG8oso1X-vif2bH/llm-project/lora-qlora-financial-sentiment
src/ found: True
Setup complete! ✅


In [2]:
# Step 2: Load Test Set Predictions for All Models
import os
import pandas as pd
import numpy as np

def load_predictions(path, method_name):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df['method'] = method_name
        return df
    print(f'Warning: {path} not found. Ensure earlier notebooks have been executed.')
    return None

zs_df = load_predictions('results/predictions/zero_shot.csv', 'Zero-Shot')
lora_df = load_predictions('results/predictions/lora.csv', 'LoRA')
qlora_df = load_predictions('results/predictions/qlora.csv', 'QLoRA')

In [3]:
# Step 3: Breakdown Error Types by Category
error_categories = [
    ('positive', 'neutral'),
    ('positive', 'negative'),
    ('negative', 'neutral'),
    ('negative', 'positive'),
    ('neutral', 'positive'),
    ('neutral', 'negative'),
]

def summarize_errors(df, name):
    if df is None:
        return {f'{true_l}->{pred_l}': 'NOT_RUN' for true_l, pred_l in error_categories}
    errors = df[df['label_text'] != df['predicted_label']]
    total_errors = len(errors)
    breakdown = {'Total Errors': total_errors}
    for true_l, pred_l in error_categories:
        count = len(errors[(errors['label_text'] == true_l) & (errors['predicted_label'] == pred_l)])
        breakdown[f'{true_l} -> {pred_l}'] = count
    return breakdown

summary = {
    'Zero-Shot': summarize_errors(zs_df, 'Zero-Shot'),
    'LoRA': summarize_errors(lora_df, 'LoRA'),
    'QLoRA': summarize_errors(qlora_df, 'QLoRA'),
}

error_summary_df = pd.DataFrame(summary)
print('=== Classification Errors by Direction ===')
display(error_summary_df)

=== Classification Errors by Direction ===


,Zero-Shot,LoRA,QLoRA
Total Errors,161,103,104
positive -> neutral,86,44,51
positive -> negative,6,1,2
negative -> neutral,5,4,5
negative -> positive,0,0,0
neutral -> positive,26,36,27
neutral -> negative,22,18,19


In [4]:
# Step 4: Examine Specific Challenging Financial Sentences
if lora_df is not None:
    errors = lora_df[lora_df['label_text'] != lora_df['predicted_label']]
    print(f'Displaying 5 actual misclassified sentences from LoRA fine-tuning:\n')
    for i, (_, row) in enumerate(errors.head(5).iterrows()):
        print(f'Example {i+1}:')
        print(f'  Sentence:  "{row["sentence"]}"')
        print(f'  True:      {row["label_text"]}')
        print(f'  Predicted: {row["predicted_label"]}')
        print('-' * 60)

Displaying 5 actual misclassified sentences from LoRA fine-tuning:

Example 1:
  Sentence:  "Under the agreement Benefon 's forthcoming range of TWIG integrated GPS navigation and mobile phone devices will use the jointly developed web-based tracking and location technology , in both consumer and commercial applications ."
  True:      neutral
  Predicted: positive
------------------------------------------------------------
Example 2:
  Sentence:  "The decision to sell a share in Lenenergo was reached in line with Fortum 's strategic plans to invest mainly in generating assets in Russia , while Lenenergo is a grid company , the press release said ."
  True:      positive
  Predicted: neutral
------------------------------------------------------------
Example 3:
  Sentence:  "It will combine and renew the contents and functionality of the current systems for funding and online service as well as their maintenance ."
  True:      positive
  Predicted: neutral
--------------------------

In [5]:
# Step 5: Common Failure Modes in Financial NLP
print('=== Key Linguistic Failure Modes Identified ===\n')
print('1. Mixed Sentiment:')
print('   Example: "Net sales increased 10%, but operating profit fell due to raw material costs."')
print('   Challenge: Model must weigh revenue growth against margin compression.\n')
print('2. Neutral Statements with Market Verbs:')
print('   Example: "The company acquired a 25% stake in the subsidiary."')
print('   Challenge: Model may falsely predict positive because "acquisition" sounds proactive.\n')
print('3. Numerical and Percentage Baselines:')
print('   Example: "Sales remained flat at EUR 50 million."')
print('   Challenge: Some models confuse static performance with negative decline.\n')

=== Key Linguistic Failure Modes Identified ===

1. Mixed Sentiment:
   Example: "Net sales increased 10%, but operating profit fell due to raw material costs."
   Challenge: Model must weigh revenue growth against margin compression.

2. Neutral Statements with Market Verbs:
   Example: "The company acquired a 25% stake in the subsidiary."
   Challenge: Model may falsely predict positive because "acquisition" sounds proactive.

3. Numerical and Percentage Baselines:
   Example: "Sales remained flat at EUR 50 million."
   Challenge: Some models confuse static performance with negative decline.



### Error Analysis Conclusions

1. **Zero-Shot vs. Fine-Tuned:** The zero-shot model makes more catastrophic errors (e.g., confusing positive for negative).
2. **Boundary Precision:** Fine-tuning with LoRA and QLoRA sharply improves sensitivity to financial terminology and reduces neutral-class ambiguity.
3. **Remaining Errors:** The remaining errors in both LoRA and QLoRA stem primarily from mixed-sentiment sentences where human financial annotators themselves frequently disagree.